# 01 · Capa Bronze y análisis exploratorio (EDA)

**Qué hace este notebook**
1. Lee el CSV del MINSA con Spark y lo guarda **tal cual** en una tabla (capa *Bronze*).
2. Explora los datos: tamaño, nulos, valores distintos, distribución por año, valores raros y duplicados.

**Idea clave:** en Bronze no se corrige nada. Si algo está mal en el origen, aquí queda a la vista y se corrige después, en Silver.

In [ ]:
%run ./00_utilidades

## 1. Leer el CSV
`inferSchema=False` deja todas las columnas como texto. Así Spark no recorre el archivo dos veces y no falla con valores raros.

In [ ]:
df_raw = spark.read.csv(RUTA_CSV, header=True, inferSchema=False, sep=";", encoding="UTF-8")
print(f"Filas: {df_raw.count():,} | Columnas: {len(df_raw.columns)}")
df_raw.printSchema()
display(df_raw.limit(5))

## 2. Guardar en Bronze
Agregamos dos columnas de auditoría: de qué archivo vino cada fila y cuándo se cargó.

In [ ]:
df_bronze = df_raw.select("*",
                          F.col("_metadata.file_name").alias("_archivo_origen"),
                          F.current_timestamp().alias("_fecha_ingesta"))
guardar_tabla(df_bronze, T_BRONZE)
print("Bronze guardada:", f"{leer_tabla(T_BRONZE).count():,} filas")

## 3. EDA · Estadísticos de las columnas numéricas
`describe()` entrega conteo, media, desviación, mínimo y máximo. Fíjate en los **mínimos y máximos**: ahí se ven los valores imposibles.

In [ ]:
numericas = ["FecNac_Año", "FecNac_Mes", "PESO_NACIDO", "TALLA_NACIDO", "DUR_EMB_PARTO", "Edad_Madre"]
df_num = df_raw.select(*[F.expr(f"try_cast(`{c}` as double)").alias(c) for c in numericas])
display(df_num.describe())

## 4. EDA · Nulos y vacíos por columna

In [ ]:
conteo = df_raw.select([F.sum(F.when(F.col(c).isNull() | (F.trim(F.col(c)) == ""), 1).otherwise(0)).alias(c)
                        for c in COLUMNAS_ORIGINALES]).collect()[0].asDict()
display(spark.createDataFrame([(k, int(v)) for k, v in conteo.items()], ["columna", "nulos_o_vacios"]))

## 5. EDA · Cuántos valores distintos tiene cada columna

In [ ]:
distintos = df_raw.select([F.approx_count_distinct(F.col(c)).alias(c) for c in COLUMNAS_ORIGINALES]).collect()[0].asDict()
display(spark.createDataFrame([(k, int(v)) for k, v in distintos.items()], ["columna", "valores_distintos"]).orderBy(F.desc("valores_distintos")))

## 6. EDA · Nacimientos por año
La forma de esta curva importa: si un año tiene muchos menos registros que otros, el archivo puede estar incompleto.

In [ ]:
display(df_raw.groupBy("FecNac_Año").count().orderBy("FecNac_Año"))

## 7. EDA · Categorías principales

In [ ]:
for c in ["Tipo_Parto", "Financiador_Parto", "Condicion_Parto", "Nivel_Intrucción_Madre"]:
    print("-", c)
    display(df_raw.groupBy(F.trim(F.col(c)).alias(c)).count().orderBy(F.desc("count")))

## 8. EDA · Valores atípicos y filas desplazadas
Cada columna cuenta cuántas filas rompen una regla razonable.

In [ ]:
e = df_raw.select(
    F.expr("try_cast(Edad_Madre as int)").alias("edad"),
    F.expr("try_cast(PESO_NACIDO as double)").alias("peso"),
    F.expr("try_cast(DUR_EMB_PARTO as int)").alias("semanas"),
    F.expr("try_cast(TALLA_NACIDO as double)").alias("talla"),
    F.trim("Financiador_Parto").alias("financiador"),
    F.trim("Lugar_Nacido").alias("lugar"),
    F.trim("Num_embar_madre").alias("embarazos"))
reglas = {
    "edad_menor_10": "edad < 10",
    "edad_mayor_55": "edad > 55",
    "peso_cero_o_negativo": "peso <= 0",
    "peso_mayor_6000": "peso > 6000",
    "semanas_mayor_45": "semanas > 45",
    "talla_fuera_20_70": "talla < 20 OR talla > 70",
    "financiador_fuera_de_lista": "financiador NOT IN ('SIS','ESSALUD','PARTICULAR','PRIVADOS','SANIDAD EP','SANIDAD FAP','SANIDAD NAVAL','SANIDAD PNP','IGNORADO')",
    "lugar_nacido_numerico": "lugar RLIKE '^[0-9]+$'",
    "num_embarazos_invalido": "NOT (embarazos RLIKE '^(-?[0-9]+|>=5)$')",
}
fila = e.agg(*[F.sum(F.when(F.expr(cond), 1).otherwise(0)).alias(k) for k, cond in reglas.items()]).collect()[0].asDict()
display(spark.createDataFrame([(k, int(v)) for k, v in fila.items()], ["regla", "filas_que_la_incumplen"]))

## 9. EDA · Duplicados exactos

In [ ]:
total = df_raw.count()
unicas = df_raw.select(*COLUMNAS_ORIGINALES).dropDuplicates().count()
print(f"Filas: {total:,} | Filas únicas: {unicas:,} | Duplicados exactos: {total - unicas:,}")

## 10. Hallazgos (complétalos con tu salida)
- El archivo tiene **1,048,575 filas**, el límite de una hoja de Excel menos el encabezado → puede estar truncado.
- No hay nulos ni vacíos, pero **sí hay filas desplazadas** (ej. `Lugar_Nacido` con números, `Num_embar_madre` con texto de ocupaciones).
- Hay edades de 0 a 60, pesos de hasta 6,590 g, semanas de gestación de hasta 99 y tallas imposibles.
- Hay **115 filas idénticas** en todos sus campos. Se conservan (no hay identificador para saber si son repetidas).